# AI4002 — Computer Vision | Semester Project Part 2 (Model 1: Runs 1–3)
## Hybrid Feature Learning: DINOv2 ViT-B/14 (LoRA) + Handcrafted Feature Fusion on CheXpert

### 1. Project Information

| Field | Details |
|---|---|
| **Group No.** | `[GROUP NUMBER]` |
| **Student Name** | `[YOUR FULL NAME]` |
| **Roll No.** | `[YOUR ROLL NUMBER]` |
| **Project Title** | Hybrid Feature Learning: Integrating Handcrafted Visual Descriptors with Advanced Vision Models for Multi-Label Chest X-Ray Classification |
| **Base Paper** | Baharoon et al. (2024), *Evaluating General Purpose Vision Foundation Models for Medical Image Analysis: An Experimental Study of DINOv2 on Radiology Benchmarks* (`arXiv:2312.02366v4`) |
| **Dataset** | CheXpert (5 competition pathologies, frontal views, 10,000-sample subset + fixed official val + deterministic held-out test) |
| **Advanced Model 1** | **DINOv2 ViT-B/14 + LoRA PEFT (`r=16, alpha=16`)** (adapts self-attention projections rather than frozen linear probing in Part 1) |
| **Feature Extractor 1** | **HOG (Histogram of Oriented Gradients)** — 441-dim spatial edge/contour descriptor |
| **Feature Extractor 2** | **Gabor Filter Bank** — 192-dim multi-scale (4 wavelengths) × multi-orientation (8 angles) texture descriptor |
| **Feature Fusion Strategy 1** | **Projection & Dimensionality Alignment + Gated Concatenation Fusion** |

### Experimental Matrix — Model 1 (Runs 1–3 of 6; Pair with `10_hybrid_fusion_model2_chexpert.ipynb` for Runs 4–6)

| Run | Configuration | Description |
|---|---|---|
| **Run 1** | `DINOv2-LoRA` (Deep Only) | Train & evaluate LoRA-adapted DINOv2 ViT-B/14 **without** additional handcrafted features |
| **Run 2** | `DINOv2-LoRA + HOG` | Integrate DINOv2-LoRA with **Feature Extractor 1 (HOG)** via Gated Projection-Concatenation Fusion |
| **Run 3** | `DINOv2-LoRA + Gabor` | Integrate DINOv2-LoRA with **Feature Extractor 2 (Gabor Filter Bank)** via Gated Projection-Concatenation Fusion |

In [ ]:
"""
Colab environment setup: clone/sync repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft/part2_chexpert"
except ImportError:
    CHECKPOINT_DIR = "outputs/part2_chexpert"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Working directory:    {os.getcwd()}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 2. Experimental Consistency & Fixed Configuration

To ensure a strictly fair comparison across **Run 1**, **Run 2**, and **Run 3** (Section 8 of Part 2 requirements), all three runs share the **exact same**:
- Dataset & train / validation / held-out test splits (`seed=42`)
- Image preprocessing & data augmentation (`224×224`, `RandomResizedCrop(0.75, 1.0)`, `RandomHorizontalFlip(0.5)`, ImageNet normalization)
- Training epochs (`10`), batch size (`32`), optimizer (`AdamW`), learning rates (`lora_lr=2e-4`, `head_lr=5e-3`, `weight_decay=1e-4`), loss (`MultiLabelBCELoss`), and random seed (`42`)
- Evaluation metrics (**Mean AUROC**, **Macro-F1**, **Macro Precision**, **Macro Recall**, and **Per-Label AUROC**)

In [ ]:
"""
Load CheXpert configuration and lock shared Part 2 hyperparameters for all 3 runs.
"""
import json
import random
import time
from typing import Any
import numpy as np
import torch
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")

PART2_CFG: dict[str, Any] = {
    "seed": 42,
    "epochs": 10,
    "batch_size": 32,
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "lora_lr": 2e-4,
    "head_lr": 5e-3,
    "weight_decay": 1e-4,
    "proj_dim": 256,
    "dropout": 0.1,
}

# Override batch size in cfg so dataloaders match PART2_CFG identically across all runs
cfg["training"]["batch_size"] = PART2_CFG["batch_size"]
LABEL_NAMES: list[str] = list(cfg["dataset"]["label_names"])
NUM_LABELS: int = len(LABEL_NAMES)

# Reference Part 1 / Base Paper results (Baharoon et al., 2024, ViT-B/14 Linear Probe)
BASE_PAPER_CHEXPERT_AUROC: float = 0.812

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Labels ({NUM_LABELS}): {LABEL_NAMES}")
print("Fixed Part 2 Hyperparameters:", json.dumps(PART2_CFG, indent=2))

## 3. Build CheXpert Datasets & DataLoaders

Construct the identical `train_loader`, `val_loader` (official 234-image validation split for model selection), and `test_loader` (deterministic 20% held-out carve-out) used across all three experimental runs.

In [ ]:
"""
Construct identical CheXpert DataLoaders shared across Run 1, Run 2, and Run 3,
and compute per-label pos_weight from the training split to handle class imbalance.
"""
from src.data import build_chexpert_dataloaders, build_transforms

set_seed(PART2_CFG["seed"])
pre = cfg["preprocessing"]
train_tf = build_transforms(
    pre["image_size"],
    pre["normalize_mean"],
    pre["normalize_std"],
    augment=True,
    crop_scale_min=pre.get("crop_scale_min", 0.75),
    crop_scale_max=pre.get("crop_scale_max", 1.0),
)
val_tf = build_transforms(
    pre["image_size"],
    pre["normalize_mean"],
    pre["normalize_std"],
    augment=False,
    resize_ratio=pre.get("resize_ratio", 1.14),
)

train_loader, val_loader, test_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)

pos_counts = torch.zeros(NUM_LABELS, dtype=torch.float64)
total_n = 0
for b in train_loader:
    pos_counts += b["labels"].sum(dim=0).double()
    total_n += b["labels"].shape[0]
POS_WEIGHT = ((total_n - pos_counts) / pos_counts.clamp(min=1.0)).clamp(1.0, 5.0).float()

print(
    f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | "
    f"Test batches: {len(test_loader)}"
)
print("Per-label pos_weight:", dict(zip(LABEL_NAMES, [round(float(w), 2) for w in POS_WEIGHT])))

## 4. Handcrafted Feature Extractors (Feature Extractor 1 & 2)

1. **Feature Extractor 1 — HOG (`HOGFeatureExtractor`):** Extracts local gradient orientation histograms over a 7×7 spatial grid (9 orientation bins → 441-dim L2-normalized descriptor). In chest radiographs, HOG explicitly captures structural boundaries such as the cardiothoracic silhouette (Cardiomegaly), costophrenic angle blunting (Pleural Effusion), and lung-field contours.
2. **Feature Extractor 2 — Gabor Filter Bank (`GaborFeatureExtractor`):** Convolves images with 32 oriented 2D Gabor wavelets (8 orientations × 4 spatial wavelengths) and pools global mean, standard deviation, and 2×2 quadrant energies (192-dim L2-normalized descriptor). Gabor responses capture multi-scale parenchymal texture patterns associated with Pulmonary Edema, Consolidation, and Atelectasis.

In [ ]:
"""
Handcrafted Feature Extractors (GPU-accelerated pure PyTorch modules):
  1. HOGFeatureExtractor   -- Histogram of Oriented Gradients (shape & edge structure)
  2. GaborFeatureExtractor -- Multi-scale, multi-orientation Gabor Filter Bank (texture)

Both modules operate directly on ImageNet-normalized (B, 3, 224, 224) tensors so they
integrate seamlessly with live data augmentation and run at GPU speed.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class HOGFeatureExtractor(nn.Module):
    """Differentiable/vectorized Histogram of Oriented Gradients (HOG) extractor.

    Computes Sobel gradients on unnormalized luminance, soft-bins unsigned gradient
    orientations into `num_bins` bins over a `(grid_size x grid_size)` spatial grid,
    and applies L2 normalization.
    Output dimension: num_bins * grid_size * grid_size (default: 9 * 7 * 7 = 441).
    """

    def __init__(self, num_bins: int = 9, grid_size: int = 7) -> None:
        super().__init__()
        self.num_bins = num_bins
        self.grid_size = grid_size
        self.out_dim = num_bins * grid_size * grid_size

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]])
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))

        bin_centers = torch.linspace(0.0, math.pi, num_bins + 1)[:-1] + (math.pi / (2.0 * num_bins))
        self.register_buffer("bin_centers", bin_centers.view(1, num_bins, 1, 1))
        self.bin_width = math.pi / num_bins

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_raw = (x.float() * self.std + self.mean).clamp(0.0, 1.0)
        gray = 0.299 * x_raw[:, 0:1] + 0.587 * x_raw[:, 1:2] + 0.114 * x_raw[:, 2:3]

        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        mag = torch.sqrt(gx * gx + gy * gy + 1e-8)
        angle = torch.atan2(gy, gx) % math.pi  # unsigned orientation in [0, pi)

        # Triangular soft voting across orientation bins
        diff = torch.abs(angle - self.bin_centers)
        diff = torch.minimum(diff, math.pi - diff)
        weights = F.relu(1.0 - diff / self.bin_width)
        hist_map = mag * weights  # (B, num_bins, H, W)

        pooled = F.adaptive_avg_pool2d(hist_map, (self.grid_size, self.grid_size))
        feat = pooled.flatten(start_dim=1)  # (B, num_bins * grid_size * grid_size)
        return F.normalize(feat, p=2, dim=-1, eps=1e-6)


class GaborFeatureExtractor(nn.Module):
    """Multi-scale, multi-orientation 2D Gabor Filter Bank texture extractor.

    Applies `len(orientations) * len(wavelengths)` fixed 2D Gabor kernels to luminance,
    then pools global mean, global std, and 2x2 spatial quadrant means of filter energy.
    Output dimension: num_filters * 6 (default: 32 * 6 = 192).
    """

    def __init__(
        self,
        num_orientations: int = 8,
        wavelengths: tuple[float, ...] = (4.0, 8.0, 12.0, 16.0),
        kernel_size: int = 15,
    ) -> None:
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        half = kernel_size // 2
        y, x = torch.meshgrid(
            torch.arange(-half, half + 1, dtype=torch.float32),
            torch.arange(-half, half + 1, dtype=torch.float32),
            indexing="ij",
        )

        kernels = []
        for lam in wavelengths:
            sigma = 0.56 * lam
            gamma = 0.5
            for k in range(num_orientations):
                theta = k * math.pi / num_orientations
                x_t = x * math.cos(theta) + y * math.sin(theta)
                y_t = -x * math.sin(theta) + y * math.cos(theta)
                gb = torch.exp(-(x_t**2 + (gamma**2) * (y_t**2)) / (2.0 * sigma**2)) * torch.cos(
                    2.0 * math.pi * x_t / lam
                )
                gb = gb - gb.mean()
                gb = gb / (gb.norm() + 1e-6)
                kernels.append(gb)

        kernel_bank = torch.stack(kernels, dim=0).unsqueeze(1)  # (F, 1, K, K)
        self.register_buffer("kernels", kernel_bank)
        self.padding = half
        self.out_dim = len(kernels) * 6

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_raw = (x.float() * self.std + self.mean).clamp(0.0, 1.0)
        gray = 0.299 * x_raw[:, 0:1] + 0.587 * x_raw[:, 1:2] + 0.114 * x_raw[:, 2:3]

        resp = torch.abs(F.conv2d(gray, self.kernels, padding=self.padding))  # (B, F, H, W)
        g_mean = resp.mean(dim=(2, 3))
        g_std = resp.std(dim=(2, 3))
        quad = F.adaptive_avg_pool2d(resp, (2, 2)).flatten(start_dim=1)  # (B, 4*F)
        feat = torch.cat([g_mean, g_std, quad], dim=-1)
        return F.normalize(feat, p=2, dim=-1, eps=1e-6)


# Quick sanity check
_dummy = torch.randn(2, 3, 224, 224)
_hog = HOGFeatureExtractor()
_gabor = GaborFeatureExtractor()
print(f"HOG output shape:   {tuple(_hog(_dummy).shape)} (dim={_hog.out_dim})")
print(f"Gabor output shape: {tuple(_gabor(_dummy).shape)} (dim={_gabor.out_dim})")

## 5. Advanced Model & Hybrid Feature-Fusion Architecture

- **Advanced Model:** `facebook/dinov2-base` (ViT-B/14) adapted with **LoRA (`r=16, alpha=16`)** on the attention query and value projections. Unlike Part 1's frozen linear probe, LoRA adapts the foundation model's internal attention geometry to radiology while keeping >98% of backbone weights frozen.
- **Feature Fusion (Projection + Dimensionality Alignment + Gated Concatenation):**
  - Deep `[CLS]` embedding (`768-dim`) → `LayerNorm → Linear(768 → 256) → GELU` (`f_deep`)
  - Handcrafted descriptor (`441-dim` HOG or `192-dim` Gabor) → `LayerNorm → Linear(d_hc → 256) → GELU` (`f_hc`)
  - Learnable gate `g = σ(Linear([f_deep, f_hc] → 256))` modulates the handcrafted representation before concatenation `[f_deep, g ⊙ f_hc] ∈ ℝ^512`, followed by the classification MLP.

In [ ]:
"""
Advanced Deep Learning Model (DINOv2 ViT-B/14 + LoRA) and Hybrid Feature-Fusion Architecture.

Supports:
  - Run 1: Deep-only (feature_extractor=None) -> DINOv2-LoRA CLS -> Projection -> Classifier
  - Run 2: Hybrid HOG (feature_extractor=HOGFeatureExtractor()) -> Projection + Gated Concatenation Fusion
  - Run 3: Hybrid Gabor (feature_extractor=GaborFeatureExtractor()) -> Projection + Gated Concatenation Fusion
"""
from typing import Any
import torch
import torch.nn as nn
from src.models import apply_lora, count_trainable_parameters, load_backbone


def build_lora_backbone(r: int = 16, lora_alpha: int = 16, lora_dropout: float = 0.1) -> nn.Module:
    """Loads DINOv2 ViT-B/14 and applies LoRA with transformers 4.x / 5.x name compatibility."""
    backbone, _ = load_backbone()
    leaf_names = {name.rsplit(".", 1)[-1] for name, _ in backbone.named_modules() if name}
    attn_map = {"query": "q_proj", "key": "k_proj", "value": "v_proj"}
    target_modules = [
        attn_map[t] if t not in leaf_names and attn_map.get(t) in leaf_names else t
        for t in ["query", "value"]
    ]
    lora_cfg: dict[str, Any] = {
        "r": r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "bias": "none",
        "target_modules": target_modules,
    }
    return apply_lora(backbone, lora_cfg)


class HybridDINOv2Model(nn.Module):
    """DINOv2 ViT-B/14 + LoRA with optional Gated Projection-Concatenation Feature Fusion."""

    def __init__(
        self,
        backbone: nn.Module,
        num_outputs: int,
        feature_extractor: nn.Module | None = None,
        proj_dim: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.feature_extractor = feature_extractor
        self.deep_proj = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, proj_dim),
            nn.GELU(),
        )

        if feature_extractor is not None:
            hc_dim: int = int(feature_extractor.out_dim)
            self.hc_proj = nn.Sequential(
                nn.LayerNorm(hc_dim),
                nn.Linear(hc_dim, proj_dim),
                nn.GELU(),
            )
            # Gated concatenation fusion: learns per-dimension weighting of handcrafted features
            self.fusion_gate = nn.Sequential(
                nn.Linear(proj_dim * 2, proj_dim),
                nn.Sigmoid(),
            )
            self.fusion_head = nn.Sequential(
                nn.LayerNorm(proj_dim * 2),
                nn.Dropout(dropout),
                nn.Linear(proj_dim * 2, proj_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(proj_dim, num_outputs),
            )
        else:
            self.hc_proj = None
            self.fusion_gate = None
            self.fusion_head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(proj_dim, num_outputs),
            )

    def head_parameters(self):
        params = list(self.deep_proj.parameters()) + list(self.fusion_head.parameters())
        if self.hc_proj is not None:
            params += list(self.hc_proj.parameters())
        if self.fusion_gate is not None:
            params += list(self.fusion_gate.parameters())
        return params

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        cls_token = self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :]
        f_deep = self.deep_proj(cls_token)

        if self.feature_extractor is not None:
            hc_raw = self.feature_extractor(pixel_values).to(f_deep.dtype)
            f_hc = self.hc_proj(hc_raw)
            gate = self.fusion_gate(torch.cat([f_deep, f_hc], dim=-1))
            fused = torch.cat([f_deep, gate * f_hc], dim=-1)
            return self.fusion_head(fused)

        return self.fusion_head(f_deep)

## 6. Unified Training & Multi-Metric Evaluation Pipeline

Evaluates every run on **4 core metrics** (plus per-label AUROC):
1. **Mean AUROC** (primary base-paper metric)
2. **Macro-F1**
3. **Macro Precision**
4. **Macro Recall**

Checkpoints are selected strictly by **Validation Mean AUROC**; final scores are reported on the **held-out Test split**.

In [ ]:
"""
Unified training and 4-metric evaluation runner ensuring 100% consistency across all 3 runs.
"""
import copy
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.optim import AdamW
from src.evaluation.metrics import compute_auroc
from src.training.losses import MultiLabelBCELoss
from src.training.schedulers import get_scheduler


@torch.no_grad()
def evaluate_multilabel(model: nn.Module, loader, device: torch.device) -> dict[str, Any]:
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        pv = batch["pixel_values"].to(device, non_blocking=True)
        lb = batch["labels"]
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(pv)
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(lb.numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0).astype(int)
    probs_np = 1.0 / (1.0 + np.exp(-logits_np))
    preds_np = (probs_np >= 0.5).astype(int)

    auroc_out = compute_auroc(labels_np, probs_np, label_names=LABEL_NAMES)
    return {
        "mean_auroc": float(auroc_out["mean_auroc"]),
        "per_label_auroc": {k: float(v) for k, v in auroc_out["per_label_auroc"].items()},
        "macro_f1": float(f1_score(labels_np, preds_np, average="macro", zero_division=0.0)),
        "macro_precision": float(precision_score(labels_np, preds_np, average="macro", zero_division=0.0)),
        "macro_recall": float(recall_score(labels_np, preds_np, average="macro", zero_division=0.0)),
    }


def run_experiment(
    run_id: str,
    run_label: str,
    feature_extractor: nn.Module | None,
) -> dict[str, Any]:
    set_seed(PART2_CFG["seed"])
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    backbone = build_lora_backbone(
        r=PART2_CFG["lora_r"],
        lora_alpha=PART2_CFG["lora_alpha"],
        lora_dropout=PART2_CFG["lora_dropout"],
    )
    model = HybridDINOv2Model(
        backbone=backbone,
        num_outputs=NUM_LABELS,
        feature_extractor=feature_extractor,
        proj_dim=PART2_CFG["proj_dim"],
        dropout=PART2_CFG["dropout"],
    ).to(device)

    trainable_params, total_params = count_trainable_parameters(model)
    print(f"\n================ {run_label} ================")
    print(
        f"Trainable params: {trainable_params:,} / {total_params:,} "
        f"({100.0 * trainable_params / total_params:.2f}%)"
    )

    criterion = MultiLabelBCELoss(pos_weight=POS_WEIGHT.to(device))
    optimizer = AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": PART2_CFG["lora_lr"]},
            {"params": model.head_parameters(), "lr": PART2_CFG["head_lr"]},
        ],
        weight_decay=PART2_CFG["weight_decay"],
    )
    total_steps = PART2_CFG["epochs"] * len(train_loader)
    scheduler = get_scheduler("cosine", optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    history: list[dict[str, float]] = []
    best_val_auroc: float = -1.0
    best_epoch: int = 0
    best_state = None
    start_time = time.time()

    for epoch in range(1, PART2_CFG["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            pv = batch["pixel_values"].to(device, non_blocking=True)
            lb = batch["labels"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(pv)
                loss = criterion(logits, lb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        val_metrics = evaluate_multilabel(model, val_loader, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_auroc": val_metrics["mean_auroc"],
                "val_macro_f1": val_metrics["macro_f1"],
            }
        )
        print(
            f"Epoch [{epoch:02d}/{PART2_CFG['epochs']:02d}] "
            f"Loss: {train_loss:.4f} | Val AUROC: {val_metrics['mean_auroc']:.4f} | "
            f"Val F1: {val_metrics['macro_f1']:.4f}"
        )
        if val_metrics["mean_auroc"] > best_val_auroc:
            best_val_auroc = val_metrics["mean_auroc"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    elapsed_sec = time.time() - start_time
    peak_mem_mb = (
        torch.cuda.max_memory_allocated(device) / (1024**2) if device.type == "cuda" else 0.0
    )

    model.load_state_dict(best_state)
    best_val = evaluate_multilabel(model, val_loader, device)
    best_test = evaluate_multilabel(model, test_loader, device)

    ckpt_path = f"{CHECKPOINT_DIR}/{run_id}_best.pt"
    torch.save(best_state, ckpt_path)

    print(f"Best Epoch: {best_epoch} | Time: {elapsed_sec:.1f}s | Peak GPU: {peak_mem_mb:.1f} MB")
    print(
        f"Test -> Mean AUROC: {best_test['mean_auroc']:.4f} | Macro-F1: {best_test['macro_f1']:.4f} | "
        f"Precision: {best_test['macro_precision']:.4f} | Recall: {best_test['macro_recall']:.4f}"
    )

    return {
        "run_id": run_id,
        "run_label": run_label,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "best_epoch": best_epoch,
        "train_time_sec": round(elapsed_sec, 1),
        "peak_gpu_mb": round(peak_mem_mb, 1),
        "val": best_val,
        "test": best_test,
        "history": history,
        "checkpoint": ckpt_path,
    }


ALL_RUNS: dict[str, dict[str, Any]] = {}

## 7. Run 1 — Advanced Model Without Additional Features (`DINOv2-LoRA`)

In [ ]:
"""
Execute Run 1: DINOv2 ViT-B/14 + LoRA (r=16) without handcrafted feature extraction.
"""
ALL_RUNS["run1_deep_only"] = run_experiment(
    run_id="run1_deep_only",
    run_label="Run 1: DINOv2-LoRA (No Extra Features)",
    feature_extractor=None,
)

## 8. Run 2 — Hybrid Fusion with Feature Extractor 1 (`DINOv2-LoRA + HOG`)

In [ ]:
"""
Execute Run 2: DINOv2 ViT-B/14 + LoRA (r=16) fused with HOG (Histogram of Oriented Gradients).
"""
ALL_RUNS["run2_hog_fusion"] = run_experiment(
    run_id="run2_hog_fusion",
    run_label="Run 2: DINOv2-LoRA + HOG Fusion",
    feature_extractor=HOGFeatureExtractor(num_bins=9, grid_size=7),
)

## 9. Run 3 — Hybrid Fusion with Feature Extractor 2 (`DINOv2-LoRA + Gabor`)

In [ ]:
"""
Execute Run 3: DINOv2 ViT-B/14 + LoRA (r=16) fused with Multi-Scale Gabor Filter Bank.
"""
ALL_RUNS["run3_gabor_fusion"] = run_experiment(
    run_id="run3_gabor_fusion",
    run_label="Run 3: DINOv2-LoRA + Gabor Fusion",
    feature_extractor=GaborFeatureExtractor(num_orientations=8, wavelengths=(4.0, 8.0, 12.0, 16.0)),
)

## 10. Results Comparison, Part 1 Base Paper Benchmark & Visualizations

Produces the exact comparison tables required by Section 9 of the Part 2 specification:
- **Table 1:** Side-by-side comparison of Run 1 vs. Run 2 vs. Run 3 across all 4 metrics (Held-Out Test Split)
- **Table 2:** Comparison of the best Part 2 hybrid model against the Part 1 Base Paper (Baharoon et al., 2024)
- **Visualizations:** Training/Validation curves, 4-metric grouped bar chart, and per-pathology AUROC breakdown

In [ ]:
"""
Compile Section 9 comparison tables, export JSON/CSV artifacts, and render publication plots.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

r1 = ALL_RUNS["run1_deep_only"]
r2 = ALL_RUNS["run2_hog_fusion"]
r3 = ALL_RUNS["run3_gabor_fusion"]

# --- Table 1: Required 3-Run Comparison Table (Test Split) ---
comparison_table = pd.DataFrame(
    [
        {
            "Metric": "Mean AUROC (Test)",
            "Run 1: DINOv2-LoRA": round(r1["test"]["mean_auroc"], 4),
            "Run 2: DINOv2-LoRA + HOG": round(r2["test"]["mean_auroc"], 4),
            "Run 3: DINOv2-LoRA + Gabor": round(r3["test"]["mean_auroc"], 4),
        },
        {
            "Metric": "Macro-F1 (Test)",
            "Run 1: DINOv2-LoRA": round(r1["test"]["macro_f1"], 4),
            "Run 2: DINOv2-LoRA + HOG": round(r2["test"]["macro_f1"], 4),
            "Run 3: DINOv2-LoRA + Gabor": round(r3["test"]["macro_f1"], 4),
        },
        {
            "Metric": "Macro Precision (Test)",
            "Run 1: DINOv2-LoRA": round(r1["test"]["macro_precision"], 4),
            "Run 2: DINOv2-LoRA + HOG": round(r2["test"]["macro_precision"], 4),
            "Run 3: DINOv2-LoRA + Gabor": round(r3["test"]["macro_precision"], 4),
        },
        {
            "Metric": "Macro Recall (Test)",
            "Run 1: DINOv2-LoRA": round(r1["test"]["macro_recall"], 4),
            "Run 2: DINOv2-LoRA + HOG": round(r2["test"]["macro_recall"], 4),
            "Run 3: DINOv2-LoRA + Gabor": round(r3["test"]["macro_recall"], 4),
        },
        {
            "Metric": "Mean AUROC (Val)",
            "Run 1: DINOv2-LoRA": round(r1["val"]["mean_auroc"], 4),
            "Run 2: DINOv2-LoRA + HOG": round(r2["val"]["mean_auroc"], 4),
            "Run 3: DINOv2-LoRA + Gabor": round(r3["val"]["mean_auroc"], 4),
        },
        {
            "Metric": "Trainable Parameters",
            "Run 1: DINOv2-LoRA": f"{r1['trainable_params']:,}",
            "Run 2: DINOv2-LoRA + HOG": f"{r2['trainable_params']:,}",
            "Run 3: DINOv2-LoRA + Gabor": f"{r3['trainable_params']:,}",
        },
    ]
)

print("=== Part 2 Comparison Table (CheXpert) ===")
print(comparison_table.to_string(index=False))

best_run = max(ALL_RUNS.values(), key=lambda x: x["test"]["mean_auroc"])
print(
    f"\nBest Part 2 Model: {best_run['run_label']} | "
    f"Test AUROC = {best_run['test']['mean_auroc']:.4f} "
    f"(vs. Base Paper ViT-B/14 Linear Probe = {BASE_PAPER_CHEXPERT_AUROC:.4f}, "
    f"Delta = {best_run['test']['mean_auroc'] - BASE_PAPER_CHEXPERT_AUROC:+.4f})"
)

# Save summary files
comparison_table.to_csv(f"{CHECKPOINT_DIR}/part2_chexpert_comparison.csv", index=False)
with open(f"{CHECKPOINT_DIR}/part2_chexpert_results.json", "w", encoding="utf-8") as f:
    json.dump(ALL_RUNS, f, indent=2)

# --- Plot 1: Validation AUROC & Training Loss Curves ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, label, color in [
    ("run1_deep_only", "Run 1: DINOv2-LoRA", "#4c72b0"),
    ("run2_hog_fusion", "Run 2: + HOG Fusion", "#dd8452"),
    ("run3_gabor_fusion", "Run 3: + Gabor Fusion", "#55a868"),
]:
    hist = pd.DataFrame(ALL_RUNS[key]["history"])
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=label, color=color)
    axes[1].plot(hist["epoch"], hist["val_auroc"], marker="s", label=label, color=color)

axes[0].set_title("Training Loss Across Runs (CheXpert)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()

axes[1].axhline(BASE_PAPER_CHEXPERT_AUROC, color="crimson", linestyle="--", label="Base Paper (0.812)")
axes[1].set_title("Validation Mean AUROC Across Runs (CheXpert)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean AUROC")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/part2_chexpert_curves.png", dpi=200)
plt.show()

# --- Plot 2: 4-Metric Grouped Bar Chart ---
metric_rows = []
for key, short_name in [
    ("run1_deep_only", "Run 1: Deep Only"),
    ("run2_hog_fusion", "Run 2: + HOG"),
    ("run3_gabor_fusion", "Run 3: + Gabor"),
]:
    t = ALL_RUNS[key]["test"]
    for m_label, m_key in [
        ("Mean AUROC", "mean_auroc"),
        ("Macro-F1", "macro_f1"),
        ("Macro Precision", "macro_precision"),
        ("Macro Recall", "macro_recall"),
    ]:
        metric_rows.append({"Run": short_name, "Metric": m_label, "Score": t[m_key]})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pd.DataFrame(metric_rows), x="Metric", y="Score", hue="Run", ax=ax)
ax.set_ylim(0.0, 1.0)
ax.set_title("CheXpert Part 2 — Held-Out Test Performance Comparison", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/part2_chexpert_metrics_bar.png", dpi=200)
plt.show()

# --- Plot 3: Per-Pathology Test AUROC Breakdown ---
per_label_rows = []
for key, short_name in [
    ("run1_deep_only", "Run 1: Deep Only"),
    ("run2_hog_fusion", "Run 2: + HOG"),
    ("run3_gabor_fusion", "Run 3: + Gabor"),
]:
    for pathology, score in ALL_RUNS[key]["test"]["per_label_auroc"].items():
        per_label_rows.append({"Run": short_name, "Pathology": pathology, "AUROC": score})

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=pd.DataFrame(per_label_rows), x="Pathology", y="AUROC", hue="Run", ax=ax)
ax.set_ylim(0.5, 1.0)
ax.set_title("CheXpert Part 2 — Per-Pathology Test AUROC", fontweight="bold")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/part2_chexpert_per_label_auroc.png", dpi=200)
plt.show()